# 09 — Interactive Figures (companion to Notebook 08)

Hover-enabled, interactive versions of the two-construct result — **AI exposure** (does the role involve AI?) vs **automatability** (could the role be automated?) — for the website and the live class demo.

The **static** presentation figures live in **Notebook 08** (`fig08_*.png`). This notebook is the lean interactive companion: it builds the **4 self-contained Plotly HTMLs** (`fig09_*.html`, full Plotly.js embedded → they open offline in any browser). It reads the same selected-model scores as nb08 and **writes only `fig09_*.html`** — it touches no nb08 / `fig08_*` artifact.

> **Caveat (carried from nb08):** these are *model-labeled* estimates, not human-coded ground truth; the model-selection validation set was a Claude-drafted audit set. The headline trends run on **n = 2,614** continuous scores.

In [1]:
# 09 — Interactive figures: builds the 4 self-contained Plotly HTMLs (fig09_*.html)
# from the same selected-model stratified sample nb08 uses. Writes only fig09_*.html.
import numpy as np, pandas as pd
import plotly.graph_objects as go
from scipy import stats
from pathlib import Path

HERE = Path.cwd()
ROOT = next((p for p in [HERE, *HERE.parents] if (p/"data"/"processed").exists()), HERE)
PROC = ROOT/"data"/"processed"
OUT  = ROOT/"output"/"figures"; OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PROC/"final_selected_model_scores_stratified_sample.csv", dtype={"job_id": str})
for c in ["year","final_exposure_score","final_automatable_score","final_exposure","final_automatable"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["year"] = df["year"].astype(int)
print(f"rows={len(df):,} | years={sorted(df.year.unique())}")

rows=2,614 | years=[np.int64(2024), np.int64(2026)]


In [2]:
# ---- shared palette + aggregates ----
INK="#16222e"; SUB="#5a6b7b"; GRID="#eef2f6"; FOOTC="#9aa7b4"
C_EXP="#1f6f8b"; C_EXP_L="#b9d4dd"; C_AUTO="#c85d3d"; C_AUTO_L="#ecc6b8"
IND_COLOR={"patent_ip":"#11405c","pharma_chem":"#2f8f8a","legal_services":"#3f7cac",
           "insurance":"#c47a3f","farming_forestry":"#977a55"}
LABEL={"patent_ip":"Patent / IP","pharma_chem":"Pharma / Chemical","legal_services":"Legal Services",
       "insurance":"Insurance","farming_forestry":"Farming & Forestry"}
FONT="Helvetica Neue, Arial, sans-serif"

piv_exp = df.pivot_table(index="industry_key",columns="year",values="final_exposure_score",aggfunc="mean")
piv_auto= df.pivot_table(index="industry_key",columns="year",values="final_automatable_score",aggfunc="mean")
ORDER=list(piv_exp[2026].sort_values(ascending=False).index)
N=len(df); corr=df["final_exposure_score"].corr(df["final_automatable_score"])
FOOT=(f"Model-labeled estimates · stratified sample n={N:,} (LinkedIn postings, 5 industries) · "
      "exposure: gpt-5.4 · automatability: gpt-5.4-mini")
print("order by 2026 exposure:", ORDER, "| r =", round(corr,3))

order by 2026 exposure: ['patent_ip', 'pharma_chem', 'legal_services', 'insurance', 'farming_forestry'] | r = -0.108


In [3]:
# ---- plotly helpers ----
# base() margins/footer are parameterized so the scatter figures can push the
# footer clear of a right-side legend. Defaults reproduce the original look.
def base(fig, title, subtitle, h=620, t=95, b=92, r=40, foot_y=-0.16):
    fig.update_layout(
        template="plotly_white", font=dict(family=FONT, color=INK, size=13),
        title=dict(text=f"<b>{title}</b><br><span style='font-size:13px;color:{SUB}'>{subtitle}</span>",
                   x=0.012, xanchor="left", y=0.95, font=dict(size=20)),
        paper_bgcolor="white", plot_bgcolor="white",
        margin=dict(l=70, r=r, t=t, b=b), height=h,
        hoverlabel=dict(bgcolor="white", bordercolor="#cdd6df", font=dict(family=FONT, size=12.5, color="#000000")),
        legend=dict(font=dict(size=11.5)),
    )
    fig.add_annotation(text=FOOT, xref="paper", yref="paper", x=0.0, y=foot_y,
                       showarrow=False, font=dict(size=10, color=FOOTC), xanchor="left", align="left")
    fig.update_xaxes(showgrid=True, gridcolor=GRID, zeroline=False, linecolor="#cdd6df")
    fig.update_yaxes(showgrid=True, gridcolor=GRID, zeroline=False, linecolor="#cdd6df")
    return fig

# Vertical legend pinned to the RIGHT of the plot — used by the scatter figures so the
# 5-industry legend (which wraps to 3 rows when horizontal) can never sit on the footer.
RIGHT_LEGEND = dict(orientation="v", x=1.015, xanchor="left", y=1.0, yanchor="top",
                    font=dict(size=11.5), bgcolor="rgba(255,255,255,0)")

def export(fig, name, modebar=False):
    # modebar off by default: these are web-embedded, the toolbar overlapped the title strip.
    fig.write_html(OUT/name, include_plotlyjs=True, full_html=True,
                   config={"displayModeBar":modebar,"displaylogo":False,"responsive":True,
                           "modeBarButtonsToRemove":["select2d","lasso2d","autoScale2d"]})
    print("wrote", name, f"({(OUT/name).stat().st_size/1e6:.1f} MB)")

### B1 · Movement map (interactive)

In [4]:
fig=go.Figure()
for key in ORDER:
    x24,y24=piv_exp.loc[key,2024],piv_auto.loc[key,2024]
    x26,y26=piv_exp.loc[key,2026],piv_auto.loc[key,2026]
    c=IND_COLOR[key]
    fig.add_trace(go.Scatter(
        x=[x24,x26], y=[y24,y26], mode="markers", name=LABEL[key], legendgroup=key,
        marker=dict(size=[14,24], color=c, symbol=["circle-open","circle"],
                    line=dict(width=[2.5,1.6], color="white"), opacity=1),
        text=[LABEL[key],LABEL[key]], customdata=[[2024],[2026]],
        hovertemplate="<b>%{text}</b><br>Year %{customdata[0]}<br>"
                      "AI exposure: %{x:.3f}<br>Automatability: %{y:.3f}<extra></extra>"))
    fig.add_annotation(x=x26,y=y26, ax=x24,ay=y24, xref="x",yref="y",axref="x",ayref="y",
                       showarrow=True, arrowhead=2, arrowsize=1.1, arrowwidth=2.4,
                       arrowcolor=c, opacity=0.9, standoff=13, startstandoff=7)
# right-side legend + footer pushed down so neither collides with the other
base(fig,"The two-construct story in one map",
     "Each arrow is one industry, 2024 → 2026 — big sideways shifts (exposure), little vertical change (automatability).",
     h=600, t=104, b=92, r=180, foot_y=-0.20)
fig.update_xaxes(title="Mean AI exposure score  →  (higher = role involves AI)", range=[0,0.215])
fig.update_yaxes(title="Mean automatability score  ↑", range=[0.248,0.378])
fig.update_layout(legend=RIGHT_LEGEND)
export(fig,"fig09_movement_map.html")
fig

wrote fig09_movement_map.html (4.9 MB)


### B2 · Per-posting scatter (interactive)

In [5]:
fig=go.Figure()
for key in ORDER:
    sub=df[df.industry_key==key]
    fig.add_trace(go.Scattergl(
        x=sub.final_exposure_score, y=sub.final_automatable_score, mode="markers",
        name=LABEL[key],
        marker=dict(size=7, color=IND_COLOR[key], opacity=0.45, line=dict(width=0)),
        customdata=np.stack([sub.title_clean.fillna("(no title)"), sub.year], axis=-1),
        hovertemplate="<b>%{customdata[0]}</b><br>"+LABEL[key]+" · %{customdata[1]}<br>"
                      "Exposure %{x:.2f} · Automatability %{y:.2f}<extra></extra>"))
fig.add_vline(x=0.5, line=dict(color="#9aa7b4",dash="dash",width=1.2))
fig.add_hline(y=0.5, line=dict(color="#9aa7b4",dash="dash",width=1.2))
for qx,qy,t in [(0.97,0.97,"exposed &<br>automatable"),(0.03,0.97,"automatable,<br>not exposed"),
                (0.97,0.03,"exposed,<br>not automatable"),(0.03,0.03,"neither")]:
    fig.add_annotation(x=qx,y=qy,text=t,showarrow=False,xanchor="right" if qx>0.5 else "left",
                       yanchor="top" if qy>0.5 else "bottom",font=dict(size=11,color="#90a4b8"))
fig.add_annotation(x=0.5,y=0.62,text=f"<b>r = {corr:+.2f}</b>",showarrow=False,
                   font=dict(size=15,color=INK),bgcolor="white",bordercolor="#cdd6df",borderwidth=1,borderpad=6)
base(fig,"Exposure and automatability are nearly independent",
     f"Every dot is one posting (n={N:,}); hover to read the role. r = {corr:+.2f} — the two constructs move almost independently.",
     h=680)
fig.update_xaxes(title="Final AI exposure score", range=[-0.02,1.02])
fig.update_yaxes(title="Final automatability score", range=[-0.02,1.02])
export(fig,"fig09_relationship_scatter.html")
fig

wrote fig09_relationship_scatter.html (5.0 MB)


### B3 · Industry comparison (interactive)

In [6]:
xlab=[LABEL[k] for k in ORDER]
e24=[piv_exp.loc[k,2024] for k in ORDER]; e26=[piv_exp.loc[k,2026] for k in ORDER]
a24=[piv_auto.loc[k,2024] for k in ORDER]; a26=[piv_auto.loc[k,2026] for k in ORDER]
fig=go.Figure()
fig.add_trace(go.Bar(x=xlab,y=e24,name="2024",marker_color=C_EXP_L,
                     hovertemplate="%{x}<br>2024: %{y:.3f}<extra></extra>"))
fig.add_trace(go.Bar(x=xlab,y=e26,name="2026",marker_color=C_EXP,
                     hovertemplate="%{x}<br>2026: %{y:.3f}<extra></extra>"))
buttons=[dict(label="AI exposure",method="update",
              args=[{"y":[e24,e26],"marker.color":[C_EXP_L,C_EXP]},
                    {"yaxis.title.text":"Mean AI exposure score","yaxis.range":[0,0.17]}]),
         dict(label="Automatability",method="update",
              args=[{"y":[a24,a26],"marker.color":[C_AUTO_L,C_AUTO]},
                    {"yaxis.title.text":"Mean automatability score","yaxis.range":[0,0.40]}])]
base(fig,"Industry comparison, 2024 vs 2026",
     "Pick a construct with the menu →", h=640)
fig.update_layout(barmode="group", bargap=0.32, bargroupgap=0.12,
                  margin=dict(l=70,r=40,t=100,b=150),
                  legend=dict(orientation="h", y=-0.34, x=0.5, xanchor="center"),
                  updatemenus=[dict(type="dropdown",buttons=buttons,x=1.0,y=1.16,xanchor="right",yanchor="top",
                                    bgcolor="white",bordercolor="#cdd6df",font=dict(size=12))])
fig.update_yaxes(title="Mean AI exposure score", range=[0,0.17])
export(fig,"fig09_industry_compare.html", modebar=False)
fig

wrote fig09_industry_compare.html (4.9 MB)


### B4 · Exposure slopegraph (interactive)

In [7]:
fig=go.Figure()
laby={k:piv_exp.loc[k,2026] for k in ORDER}   # declutter right-label y-positions
for a,b in zip(sorted(ORDER,key=lambda k:laby[k])[:-1],sorted(ORDER,key=lambda k:laby[k])[1:]):
    if laby[b]-laby[a]<0.0075: laby[b]=laby[a]+0.0075
for key in ORDER:
    y24,y26=piv_exp.loc[key,2024],piv_exp.loc[key,2026]
    c=IND_COLOR[key]; mult=y26/y24 if y24>0 else np.nan
    fig.add_trace(go.Scatter(
        x=[0,1], y=[y24,y26], mode="lines+markers", name=LABEL[key],
        line=dict(color=c,width=3.2), marker=dict(size=[9,15],color=c,line=dict(width=1.4,color="white")),
        customdata=[[2024,mult],[2026,mult]],
        hovertemplate="<b>"+LABEL[key]+"</b><br>%{customdata[0]}: %{y:.3f}<br>×%{customdata[1]:.2f} vs 2024<extra></extra>"))
    fig.add_annotation(x=1.05,y=laby[key],xref="x",yref="y",
                       text=f"<b>{LABEL[key]}</b>  {y26:.3f} ({mult:.1f}×)",
                       showarrow=False,xanchor="left",font=dict(size=11.5,color=c))
base(fig,"AI exposure diverges sharply by industry, 2024 → 2026",
     "Knowledge work surges (Patent/IP, Legal, Pharma); manual & clerical roles hold flat or dip.", h=620)
fig.update_xaxes(tickmode="array",tickvals=[0,1],ticktext=["2024","2026"],
                 range=[-0.18,2.15],showgrid=False,title="")
fig.update_yaxes(title="Mean AI exposure score", range=[0,0.17])
fig.update_layout(showlegend=False, margin=dict(l=70,r=215,t=95,b=92))
export(fig,"fig09_exposure_by_industry_slope.html")
fig

wrote fig09_exposure_by_industry_slope.html (4.9 MB)


### B5 · Posting-level density & distribution shift (interactive)

In [8]:
# Interactive joint density: contour of all postings + hoverable points.
_d = df.dropna(subset=["final_exposure_score","final_automatable_score"]).copy()
_N = len(_d); _r = float(np.corrcoef(_d.final_exposure_score, _d.final_automatable_score)[0, 1])
_rng = np.random.default_rng(42)
_d["_jx"] = np.clip(_d.final_exposure_score + _rng.normal(0, 0.012, _N), 0, 1)
_d["_jy"] = np.clip(_d.final_automatable_score + _rng.normal(0, 0.018, _N), 0, 1)
fig = go.Figure()
fig.add_trace(go.Histogram2dContour(x=_d._jx, y=_d._jy,
    colorscale=[[0,"#ffffff"],[0.25,"#cfe3ea"],[0.6,"#7fb4c4"],[1,"#11405c"]],
    showscale=False, ncontours=14, line=dict(width=0), hoverinfo="skip", opacity=0.9))
for key in [k for k in ORDER if k in _d.industry_key.unique()]:
    s = _d[_d.industry_key == key]
    fig.add_trace(go.Scattergl(x=s._jx, y=s._jy, mode="markers", name=LABEL.get(key, key),
        marker=dict(size=5, color=IND_COLOR.get(key, "#888"), opacity=0.35, line=dict(width=0)),
        customdata=np.stack([s.title_clean.fillna("(no title)"), s.year], axis=-1),
        hovertemplate="<b>%{customdata[0]}</b><br>"+LABEL.get(key, key)+" · %{customdata[1]}<br>"
                      "Exposure %{x:.2f} · Automatability %{y:.2f}<extra></extra>"))
fig.add_vline(x=0.5, line=dict(color="#9aa7b4", dash="dash", width=1.1))
fig.add_hline(y=0.5, line=dict(color="#9aa7b4", dash="dash", width=1.1))
fig.add_annotation(x=0.5, y=0.6, text=f"<b>r = {_r:+.2f}</b><br><span style='font-size:11px;color:{SUB}'>n = {_N:,} postings</span>",
    showarrow=False, font=dict(size=16, color=INK), bgcolor="white", bordercolor="#cdd6df", borderwidth=1, borderpad=6)
for qx, qy, t in [(0.97,0.97,"exposed &<br>automatable"),(0.03,0.97,"automatable,<br>not exposed"),
                  (0.97,0.03,"exposed,<br>not automatable"),(0.03,0.03,"neither")]:
    fig.add_annotation(x=qx, y=qy, text=t, showarrow=False, xanchor="right" if qx>0.5 else "left",
                       yanchor="top" if qy>0.5 else "bottom", font=dict(size=10.5, color="#90a4b8"))
# right-side legend keeps the 5 industries clear of the bottom footer
base(fig, "Exposure and automatability are nearly independent",
     f"Each point is one job posting (n={_N:,}); AI exposure barely predicts how automatable a role is (r = {_r:+.2f}). Hover any point.", h=600, t=104, b=92, r=180, foot_y=-0.20)
fig.update_xaxes(title="AI exposure score  →", range=[-0.02, 1.02])
fig.update_yaxes(title="Automatability score  ↑", range=[-0.02, 1.02])
fig.update_layout(legend=RIGHT_LEGEND)
export(fig, "fig09_joint_density.html")
fig

wrote fig09_joint_density.html (5.1 MB)


In [9]:
# Interactive ECDF with a construct dropdown.
def _ecdf(v):
    v = np.sort(v); return np.concatenate([[0], v, [1]]), np.concatenate([[0], np.arange(1, len(v)+1)/len(v), [1]])
fig = go.Figure()
_specs = [("final_exposure_score","AI exposure",C_EXP_L,C_EXP),("final_automatable_score","Automatability",C_AUTO_L,C_AUTO)]
for ci, (col, name, cl, cd) in enumerate(_specs):
    for yr, c in [(2024, cl), (2026, cd)]:
        xs, ys = _ecdf(df[df.year == yr][col].dropna().values); mu = df[df.year == yr][col].mean()
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=c, width=3, shape="hv"),
            name=f"{yr} (μ={mu:.3f})", visible=(ci == 0),
            hovertemplate=f"{name} · {yr}<br>score ≤ %{{x:.2f}}<br>share %{{y:.0%}}<extra></extra>"))
_vis_exp = [True, True, False, False]; _vis_auto = [False, False, True, True]
_buttons = [dict(label="AI exposure", method="update", args=[{"visible": _vis_exp}, {"xaxis.title.text": "AI exposure score"}]),
            dict(label="Automatability", method="update", args=[{"visible": _vis_auto}, {"xaxis.title.text": "Automatability score"}])]
# SHORT title/subtitle so the top-right dropdown never sits on the text; bottom legend + footer separated
base(fig, "Exposure climbs, automatability holds",
     "Cumulative share of postings by score (further right = higher).",
     h=620, t=116, b=158, r=40, foot_y=-0.36)
fig.update_xaxes(title="AI exposure score", range=[0, 1]); fig.update_yaxes(title="cumulative share of postings", range=[0, 1.02])
fig.update_layout(
    legend=dict(orientation="h", y=-0.16, x=0.5, xanchor="center", font=dict(size=11.5)),
    updatemenus=[dict(type="dropdown", buttons=_buttons, x=1.0, y=1.12, xanchor="right", yanchor="bottom",
                      bgcolor="white", bordercolor="#cdd6df", font=dict(size=12))])
export(fig, "fig09_distribution_shift_ecdf.html")
fig

wrote fig09_distribution_shift_ecdf.html (5.0 MB)
